# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR⁲ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

We will reference all dataset entities (record sets, fields, columns) by their `@id` to ensure reproducibility and clarity in processing.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

This dataset comprises ordered logistic regression outputs for predictors of adoption of indigenous and modern knowledge in rangeland management in Northern Kenya.


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
We load the dataset metadata and records via the Croissant schema URL using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, their `@id`, field names and their `@id`. We use these `@id` fields to access and process the data throughout the notebook.

In [ ]:
# List all record sets by @id with their fields and field @id.
print("Available record sets in this dataset:\n")
record_sets = []

for record_set in dataset.record_sets:
    print(f"- Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    record_sets.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field name: {field.name}")
        print(f"      @id: {field.id}")
    print()

# Preview a few records from each record set
for rs_id in record_sets:
    print(f"Sample records for record set {rs_id}:")
    samples = [rec for i, rec in enumerate(dataset.records(record_set=rs_id)) if i < 2]
    if samples:
        print(json.dumps(samples, indent=2))
    else:
        print("  (No sample records available)")
    print('-' * 60)


## 3. Data Extraction
Load data from each record set into a pandas DataFrame. Each record set and its fields are referenced by their `@id`.

In [ ]:
# Prepare all record sets for extraction
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set {rs_id}:")
        print(f"  Columns ({len(df.columns)}): {list(df.columns)}")
        print(df.head(2).to_string(index=False))
        print()
    else:
        print(f"No records found for {rs_id}.")

# Choose one record set for further EDA (pick the first one with records)
main_record_set_id = None
for rs_id in record_sets:
    if rs_id in dataframes:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"\nUsing record set: {main_record_set_id} for EDA")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record set found.")


## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering by threshold, normalization, and grouping—referencing fields by their `@id`. Update the numeric and group field to concrete `@id`s present in the selected record set.

In [ ]:
# Identify candidate numeric and group field from the main_record_set_id
df = dataframes.get(main_record_set_id)

# Attempt to find numeric and group fields automatically
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col].dropna())]
group_field_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col].dropna()) or pd.api.types.is_categorical_dtype(df[col].dropna())]

print(f"Numeric fields in record set {main_record_set_id}: {numeric_field_candidates}")
print(f"Group/categorical fields in record set {main_record_set_id}: {group_field_candidates}")

if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # Use first detected numeric field as an example
    print(f"Using numeric field '@id': {numeric_field_id}")
else:
    print("No numeric field found for EDA.")
    numeric_field_id = df.columns[0] if df.shape[1] else None

if group_field_candidates:
    group_field_id = group_field_candidates[0]  # Use first detected group field
    print(f"Using group field '@id': {group_field_id}")
else:
    group_field_id = None
    print("No group field available for grouping in EDA.")

# Simple numeric filter/exploration
if numeric_field_id and numeric_field_id in df.columns:
    try:
        threshold = df[numeric_field_id].dropna().quantile(0.75)  # Use 75th percentile as threshold
    except Exception:
        threshold = 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in '{main_record_set_id}' with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize selected numeric field
    filtered_df = filtered_df.copy()
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std(ddof=0)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std > 0 else 0
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field (if available)
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().reset_index()
        grouped_df = grouped_df.sort_values(by=numeric_field_id, ascending=False)
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}' in '{main_record_set_id}':")
        display(grouped_df.head())
else:
    print("Unable to compute EDA: no numeric field found.")


## 5. Visualization
Visualize the distribution of the chosen numeric field and the group comparison (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for normalized numeric field
if numeric_field_id and f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of Normalized Field: {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Count")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Comparison bar plot if grouping available
if group_field_id and 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10,5))
    sns.barplot(y=group_field_id, x=numeric_field_id, data=grouped_df)
    plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.tight_layout()
    plt.show()


## 6. Conclusion
In this notebook, we:

- Loaded the FAIR⁲ rangeland management dataset via its Croissant schema, referencing all data entities by their canonical `@id` fields for reproducibility.
- Reviewed available record sets, fields, and example records.
- Extracted full data from a selected record set and performed fundamental exploratory data analysis, including filtering, normalization, and grouping by a key attribute.
- Visualized distributions and groupwise means to illustrate potential research questions and dataset characteristics.

To extend this analysis, examine other record sets, join fields across sets by shared keys, or apply domain-specific feature engineering and statistical modeling using the precise identifiers provided in the Croissant schema.
